# ShowerLM showers: 2D and 3D views

Reads the `sample_XXXXX.npz` files written by `generate.py` (cache voxel format) and draws, in the style of the
sample gallery: the x–depth and y–depth projections (log10 photons per pixel), the photons per 0.5 ns, and a 3D view of
the brightest voxels. Set `FOLDER` to your output folder. A CORSIKA 8 cache event (`event_XXXXX.npz`) can be drawn
the same way.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

FOLDER = Path("showers")          # output folder of generate.py
SHOW = 6                          # showers per figure
XY, DEP = 3.0, 14.0               # view: |x|, |y| <= 3 m, depth 0-14 m below the injection point
G, RANGES = 960, np.array([[-10, 10], [-10, 10], [30, 50]], float)   # the 0.6 cube: 960^3 voxels of 2.083 cm

def load(path):
    """one shower in metres: x, y, depth below the injection point (0, 0, 50 m), photons q, voxel time t (ns)"""
    with np.load(path) as d:
        xyz = RANGES[:, 0] + (d["ijk"] + 0.5 + d["off"]) * (RANGES[:, 1] - RANGES[:, 0]) / G
        return dict(x=xyz[:, 0], y=xyz[:, 1], d=RANGES[2, 1] - xyz[:, 2], q=d["q"].astype(float), t=d["t"].astype(float),
                    name=Path(path).stem)

files = sorted(FOLDER.glob("sample_*.npz"))
showers = [load(f) for f in files[:SHOW]]
for s in showers:
    print(f"{s['name']}: {len(s['q'])} lit voxels, {s['q'].sum():.3e} photons")

In [ ]:
def inbox(P):
    return (np.abs(P["x"]) <= XY) & (np.abs(P["y"]) <= XY) & (P["d"] >= 0) & (P["d"] <= DEP)

def fig2d(panels):
    n = len(panels)
    fig, ax = plt.subplots(3, n, figsize=(2.15 * n, 8.2), facecolor="#0f1216", squeeze=False,
                           gridspec_kw=dict(height_ratios=[1.6, 1.6, 1.0]))
    ex, ed, et = np.linspace(-XY, XY, 121), np.linspace(0, DEP, 281), np.linspace(0, 80, 161)
    vmax = None
    for j, P in enumerate(panels):
        b = inbox(P)
        for row, key in ((0, "x"), (1, "y")):
            H = np.histogram2d(P["d"][b], P[key][b], bins=[ed, ex], weights=P["q"][b])[0]
            L = np.log10(np.maximum(H, 1))
            if vmax is None:
                vmax = float(np.percentile(L[L > 0], 99.9)) if (L > 0).any() else 7
            a = ax[row, j]
            a.imshow(L, extent=[-XY, XY, DEP, 0], aspect="auto", cmap="viridis", vmin=2, vmax=vmax, interpolation="nearest")
            a.set_facecolor("black"); a.tick_params(colors="#9aa2ad", labelsize=6)
            a.set_xlabel(f"{key} (m)", color="#9aa2ad", fontsize=7)
            if j == 0:
                a.set_ylabel("depth below injection (m)", color="#9aa2ad", fontsize=7)
            if row == 0:
                a.set_title(P["name"], color="#e6e8eb", fontsize=7)
        a = ax[2, j]
        h = np.histogram(P["t"], et, weights=P["q"])[0]
        a.step(et[:-1], np.maximum(h, 1), where="post", color="#5ec962", lw=0.9)
        a.set_yscale("log"); a.set_ylim(1e2, 2e8); a.set_facecolor("#05070a"); a.tick_params(colors="#9aa2ad", labelsize=6)
        a.set_xlabel("voxel time t (ns)", color="#9aa2ad", fontsize=7)
        if j == 0:
            a.set_ylabel("photons / 0.5 ns", color="#9aa2ad", fontsize=7)
        for s in a.spines.values():
            s.set_color("#333")
    fig.tight_layout()
    return fig

fig2d(showers);

## 3D view

The brightest 40,000 voxels in the view box of each shower, coloured by ln(photons). Change `azim` / `elev` to rotate.

In [ ]:
def fig3d(panels, npts=40000, elev=12, azim=-60):
    n = len(panels); cols = min(n, 3); rows = (n + cols - 1) // cols
    fig = plt.figure(figsize=(4.2 * cols, 5.4 * rows), facecolor="black")
    lq = np.log(np.concatenate([P["q"] for P in panels]) + 0.001)
    lo, hi = np.quantile(lq, 0.002), np.quantile(lq, 0.999)
    for k, P in enumerate(panels):
        ax = fig.add_subplot(rows, cols, k + 1, projection="3d", facecolor="black")
        b = np.flatnonzero(inbox(P)); s = b[np.argsort(P["q"][b])[::-1][:npts]]; s = s[np.argsort(P["q"][s])]
        ax.scatter(P["x"][s], P["y"][s], P["d"][s], c=np.log(P["q"][s] + 0.001), cmap="viridis", vmin=lo, vmax=hi,
                   s=0.4, alpha=0.8, linewidths=0, depthshade=False)
        ax.set_xlim(-XY, XY); ax.set_ylim(-XY, XY); ax.set_zlim(DEP, 0); ax.set_box_aspect((1, 1, 2.2))
        ax.set_xlabel("x (m)", color="#e66767", fontsize=7); ax.set_ylabel("y (m)", color="#3fbf8a", fontsize=7)
        ax.set_zlabel("depth (m)", color="#5b9cf0", fontsize=7); ax.tick_params(colors="w", labelsize=5)
        for a in (ax.xaxis, ax.yaxis, ax.zaxis):
            a.set_pane_color((0, 0, 0, 1)); a._axinfo["grid"]["color"] = (0.25, 0.25, 0.25, 1)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(P["name"], color="w", fontsize=8)
    return fig

fig3d(showers);

## Rotatable 3D (optional, needs plotly)

`pip install plotly`; drag to rotate, scroll to zoom.

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    print("plotly is not installed: pip install plotly")
else:
    P = showers[0]
    b = np.flatnonzero(inbox(P)); s = b[np.argsort(P["q"][b])[::-1][:40000]]
    go.Figure(go.Scatter3d(x=P["x"][s], y=P["y"][s], z=-P["d"][s], mode="markers",
                           marker=dict(size=1.2, color=np.log10(P["q"][s] + 1e-3), colorscale="Viridis", opacity=0.8,
                                       colorbar=dict(title="log10 q"))),
              layout=dict(title=P["name"], template="plotly_dark", height=700,
                          scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="-depth (m)", aspectmode="manual",
                                     aspectratio=dict(x=1, y=1, z=2.2)))).show()

## Per-shower numbers

`timing.csv` has one row per shower (lengths, plan totals, seconds); `gcheck.csv`, `zcheck.csv` and `tcheck.csv` compare
what the plan asked for (`want_*`) with what the shower got (`got_*`).

In [ ]:
import csv
rows = list(csv.DictReader(open(FOLDER / "timing.csv")))
for r in rows[:SHOW]:
    print(f"#{r['index']}: skeleton {r['n_skel']} strokes ({r['skel_stop']}), detail {r['n_det']} strokes, {int(r['n_vox'])} voxels, "
          f"{float(r['q_total']):.3e} photons, {float(r['total_s']):.1f} s")